# 第6部分：PSI稳定性监控

**目的：** 监控变量分布是否随时间"漂移"了

## 什么是分布漂移？

举例：你用2024年1-6月的数据训练了模型。
如果到了2025年，客户的收入分布完全变了（比如经济下行导致整体收入降低），
那模型的预测就不准了——因为它学到的"规律"已经不适用了。

## PSI = Population Stability Index（人群稳定性指数）

对比"基准期"和"观察期"的分布差异。

| PSI值 | 含义 | 行动 |
|-------|------|------|
| < 0.1 | 稳定 | 不用管 |
| 0.1 ~ 0.25 | 轻微漂移 | 需要关注 |
| > 0.25 | 严重漂移 | 必须重训模型！ |

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Union

## 步骤1：计算单个特征的PSI

### PSI公式

```
PSI = sum( (actual% - expected%) * ln(actual% / expected%) )
```

### 通俗理解

```
把特征分成10个箱
比较"基准月"和"当前月"在每个箱里的人数占比

基准月(1月)          当前月(6月)
箱1: 10%             箱1: 10%     一样
箱2: 10%             箱2: 25%     大增！
箱3: 10%             箱3:  2%     大减！
...                  ...          差异越大，PSI越大
```

In [ ]:
def calculate_feature_psi(
    df: pd.DataFrame,
    month_col: str,
    feature_list: List[str],
    base_month: Union[str, None] = None,
    bins: int = 10,
    verbose: bool = True
) -> pd.DataFrame:
    """
    计算一组特征在不同月份相对于基准月的PSI

    参数：
        df: 包含数据的DataFrame
        month_col: 月份列的列名（比如 'apply_month'）
        feature_list: 要计算PSI的特征列表
        base_month: 基准月（如果不指定，用数据中最早的月份）
        bins: 分箱数（默认10）

    返回：
        DataFrame，每行 = (月份, 变量名, PSI值)
    """

    def is_numeric_series(series):
        return pd.api.types.is_numeric_dtype(series)

    def calculate_single_psi(expected, actual, bins):
        """
        计算单个特征的PSI
        为什么加1e-6？防止某箱人数=0时log(0)报错
        为什么用基准月的分位数做分箱？固定参照系，才能公平比较
        """
        expected = expected.dropna()
        actual = actual.dropna()

        if len(expected) == 0 or len(actual) == 0:
            return np.nan

        # 用基准月的分位数作为分箱边界（固定参照系）
        breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
        breakpoints[-1] += 1e-6  # 确保最大值不被遗漏

        # 统计每箱的样本占比
        expected_hist = np.histogram(expected, bins=breakpoints)[0] + 1e-6
        actual_hist = np.histogram(actual, bins=breakpoints)[0] + 1e-6

        expected_pct = expected_hist / expected_hist.sum()
        actual_pct = actual_hist / actual_hist.sum()

        # PSI公式
        psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
        return psi

    # ---- 主逻辑 ----
    if base_month is None:
        base_month = df[month_col].min()
        if verbose:
            print(f"自动使用最早月份作为基准: {base_month}")

    base_data = df[df[month_col] == base_month]

    skipped_vars = set()
    psi_results = []

    # 遍历每个月份 x 每个特征
    for month in df[month_col].unique():
        if month == base_month:
            continue  # 不和自己比

        current_data = df[df[month_col] == month]

        for var in feature_list:
            if not is_numeric_series(base_data[var]):
                skipped_vars.add(var)
                continue

            psi = calculate_single_psi(
                expected=base_data[var],
                actual=current_data[var],
                bins=bins
            )
            psi_results.append({
                'month': month,
                'variable': var,
                'psi': psi,
                'base_month': base_month
            })

    if verbose and skipped_vars:
        print(f"已跳过 {len(skipped_vars)} 个非数值型变量")

    return pd.DataFrame(psi_results)

## 步骤2：实际使用 - 筛选稳定变量

In [ ]:
# 计算PSI（只看有表现期的、2025年1月之后的数据）
psi_df = calculate_feature_psi(
    df=df[(df['is_perform_dob2_ever30'] > 0) & (df['trans_month'] >= '202501')].copy(),
    month_col='trans_month',
    feature_list=pboc_feature,  # 要监控的特征列表
    bins=5  # 分5箱（数据量不大时箱数少一点更稳定）
)

# 筛选稳定变量：取每个变量在所有月份中PSI的最大值 < 0.1
stable_vars = psi_df.groupby('variable')['psi'].max().reset_index()
stable_vars = stable_vars[stable_vars['psi'] < 0.1]['variable'].tolist()

print(f"稳定变量数量: {len(stable_vars)}")
print(f"\n为什么取max？")
print("如果一个变量在某个月PSI>0.1，说明它偶尔不稳定")
print("这样的变量上线后可能突然失效，不能用")

## 步骤3：Kendall Tau趋势稳定性检验

PSI看的是分布整体是否漂移，Kendall Tau看的是**排序是否稳定**。

### 为什么还需要看排序？

```
信用分分5箱，正常情况：
    1月: 箱1坏账15% > 箱2坏账10% > 箱3坏账7% > 箱4坏账4% > 箱5坏账2%
    2月: 箱1坏账14% > 箱2坏账9%  > 箱3坏账6% > 箱4坏账3% > 箱5坏账1%
    排序一致！低分高风险，逻辑稳定

异常情况：
    3月: 箱3坏账12% > 箱1坏账10% > 箱5坏账8% > ...
    排序翻转了！变量的区分逻辑变了
```

### Kendall Tau值含义

| tau值 | 含义 |
|-------|------|
| >= 0.7 | 强稳定（排序高度一致） |
| 0.4 ~ 0.7 | 中等稳定 |
| < 0.4 | 不稳定（排序会翻转，变量不可靠） |

In [ ]:
from scipy.stats import kendalltau


def check_trend_stability(pivot_table, threshold=0.4):
    """
    检查分箱坏账率的排序是否跨月稳定

    做法：
        1. 每月按分箱排序坏账率
        2. 计算月份间排序的Kendall Tau相关
        3. 平均Tau > 0.4 -> 稳定

    参数：
        pivot_table: 行=分箱, 列=月份, 值=坏账率
        threshold: 稳定性阈值（默认0.4）

    返回：
        dict: avg_similarity(平均相似度), is_stable(是否稳定)
    """
    monthly_rankings = {}

    for month in pivot_table.columns:
        # rank(): 把坏账率转换成排名
        monthly_rankings[month] = pivot_table[month].rank(method='dense').values

    # 两两月份比较Kendall Tau
    similarity_scores = []
    months = list(monthly_rankings.keys())

    for i in range(len(months)):
        for j in range(i + 1, len(months)):
            tau, _ = kendalltau(
                monthly_rankings[months[i]],
                monthly_rankings[months[j]]
            )
            similarity_scores.append(tau)

    avg_similarity = np.mean(similarity_scores)
    is_stable = avg_similarity >= threshold

    return {
        'avg_similarity': avg_similarity,
        'is_stable': is_stable
    }

## 完整监控流程总结

```
变量入模前的稳定性筛选：

全部候选变量(300个)
    |
    +-- PSI筛选：去掉分布漂移的 -> 剩200个
    |
    +-- Kendall Tau筛选：去掉排序不稳定的 -> 剩150个
    |
    +-- 最终入模变量(150个)
```

---
### 面试考点

| 问题 | 答案 |
|------|------|
| PSI>0.25意味着什么？ | 变量分布严重偏移，模型可能失效，需重训 |
| 为什么用基准月的分位数做分箱？ | 固定参照系，否则每月分箱不同无法对比 |
| Kendall Tau和PSI有什么区别？ | PSI看分布是否变化，Tau看排序是否翻转 |
| 什么时候用PSI vs Tau？ | 都要用！PSI=分布稳定性，Tau=区分力稳定性 |
| 变量PSI<0.1但Tau<0.4怎么办？ | 不能用！分布没变但排序乱了=区分逻辑失效 |